In [ ]:
# AIHW Suicide Mortality Data: Data Cleaning

**Input:** Raw Excel file (five sheets)  
**Output:** Five clean pandas DataFrames, each in long format, ready for analysis

## What this notebook does and why

The exploration notebook (01) showed us that AIHW structured this data for human 
readers in Excel, wide format, multi-row headers, suppression codes as strings. 
Before we can analyse or visualise anything, we need to reshape each table into 
a format pandas can work with.

Every cleaning decision in this notebook is documented with a reason. This matters 
in health data work because cleaning choices can introduce bias if made carelessly,  
for example, replacing suppressed `'n.p.'` values with zero would artificially 
deflate rates in remote areas where small counts are common.

## 1. Setup

In [4]:
import pandas as pd
import numpy as np

FILE_PATH = 'Data/AIHW_suicide-and-self-harm-monitoring_NMD_suicide_2024.xlsx'

# These are the string values AIHW uses to represent missing or suppressed data.
# We will convert ALL of them to np.nan so pandas treats them as proper missing values.
# The distinction between '. .' (not available) and 'n.p.' (suppressed small count)
# is documented in comments throughout, but both become NaN numerically.
SUPPRESSION_CODES = ['. .', 'n.p.', 'n.a.']

print('Setup complete.')

Setup complete.


In [ ]:
## 2. Cleaning NMD 1: National trend by sex (1907–2024)

### What we're doing here

NMD 1 is in wide format, each year is its own column, running from 1907 to 2024. 
We need to convert this to long format where each row represents one observation: 
one sex, one measure type, one year, one value.

The pandas function that does this conversion is `pd.melt()`. Think of it like 
folding a wide table downward, instead of 118 year columns, you get two new 
columns called Year and Value and each row is one data point.

We will also filter to only age-standardised rates. The exploration notebook showed 
that crude rates are unavailable before 1968 and for our purposes (comparing across 
time and sex) age-standardised rates are the correct measure to use.

In [5]:
# Load with header=1: this tells pandas that row index 1 (the second row)
# contains the column names. Row 0 is the table title, which we skip.
df_nmd1_raw = pd.read_excel(FILE_PATH, sheet_name='Table NMD 1', header=1)

print('Raw shape:', df_nmd1_raw.shape)
print('\nFirst 3 rows, first 6 columns:')
print(df_nmd1_raw.iloc[:3, :6].to_string())
print('\nMeasure column unique values:')
print(df_nmd1_raw['Measure'].unique())

Raw shape: (22, 120)

First 3 rows, first 6 columns:
       Sex                              Measure       1907       1908       1909       1910
0    Males  Age-standardised rate (per 100,000)  26.694265  25.347353  25.010777  26.719395
1  Females  Age-standardised rate (per 100,000)   5.222553   4.666333   5.307562   4.935708
2  Persons  Age-standardised rate (per 100,000)  16.897077    15.9156  15.924692  16.638067

Measure column unique values:
['Age-standardised rate (per 100,000)' 'Crude rate (per 100,000)' 'Number'
 'Median age at death (years)' 'Rate ratio (male:female)' nan]


In [6]:
# Step 1: Keep only age-standardised rates.
# This is the appropriate measure for comparing across time periods and sexes
# because it controls for changes in Australia's age structure over 117 years.
df_nmd1 = df_nmd1_raw[
    df_nmd1_raw['Measure'] == 'Age-standardised rate (per 100,000)'
].copy()

# Step 2: Keep only the three summary sex categories.
# The raw table contains 'Males', 'Females', and 'Persons' (combined).
# We keep all three as they serve different analytical purposes.
df_nmd1 = df_nmd1[df_nmd1['Sex'].isin(['Males', 'Females', 'Persons'])]

print('After filtering, shape:', df_nmd1.shape)
print('Sex values present:', df_nmd1['Sex'].unique())

After filtering, shape: (3, 120)
Sex values present: ['Males' 'Females' 'Persons']


In [7]:
# Step 3: Identify the year columns.
# Everything except 'Sex' and 'Measure' is a year column.
# pandas read the year headers as integers, so we select integer column names.
id_cols = ['Sex', 'Measure']  # columns that identify what the row represents
year_cols = [c for c in df_nmd1.columns if isinstance(c, int)]  # all year columns

print(f'Number of year columns: {len(year_cols)}')
print(f'Year range: {year_cols[0]} to {year_cols[-1]}')

Number of year columns: 118
Year range: 1907 to 2024


In [8]:
# Step 4: Melt from wide to long format.
#
# pd.melt() is the key transformation here. Here's what each argument means:
#   id_vars    = columns that stay as-is (they identify the row)
#   value_vars = columns to 'melt down' (each year becomes a row)
#   var_name   = what to call the new column that holds the old column names (Year)
#   value_name = what to call the new column that holds the values (Rate)
#
# Before melt: 3 rows x 120 columns (Sex, Measure, 1907, 1908 ... 2024)
# After melt:  354 rows x 4 columns (Sex, Measure, Year, Rate)

df_nmd1_long = pd.melt(
    df_nmd1,
    id_vars=id_cols,
    value_vars=year_cols,
    var_name='Year',
    value_name='Rate'
)

print('After melt, shape:', df_nmd1_long.shape)
print('\nFirst 6 rows after melt:')
print(df_nmd1_long.head(6).to_string())

After melt, shape: (354, 4)

First 6 rows after melt:
       Sex                              Measure  Year       Rate
0    Males  Age-standardised rate (per 100,000)  1907  26.694265
1  Females  Age-standardised rate (per 100,000)  1907   5.222553
2  Persons  Age-standardised rate (per 100,000)  1907  16.897077
3    Males  Age-standardised rate (per 100,000)  1908  25.347353
4  Females  Age-standardised rate (per 100,000)  1908   4.666333
5  Persons  Age-standardised rate (per 100,000)  1908    15.9156


In [9]:
# Step 5: Handle suppression codes and fix data types.
#
# Replace '. .' (not available) with NaN.
# These appear in early years where age-standardised rates weren't calculated.
# We are NOT replacing 'n.p.' here because NMD1 doesn't use it — but the
# replace call below covers both to be safe.
for code in SUPPRESSION_CODES:
    df_nmd1_long['Rate'] = df_nmd1_long['Rate'].replace(code, np.nan)

# Convert Rate to numeric. 'coerce' means anything that can't be converted
# to a number becomes NaN rather than throwing an error.
df_nmd1_long['Rate'] = pd.to_numeric(df_nmd1_long['Rate'], errors='coerce')

# Year is already an integer from the Excel headers, but let's confirm
df_nmd1_long['Year'] = df_nmd1_long['Year'].astype(int)

# Step 6: Drop the Measure column (we only kept one measure type, so it's redundant)
# and add a source label so we know which table this came from later
df_nmd1_long = df_nmd1_long.drop(columns=['Measure'])
df_nmd1_long['Source'] = 'NMD1'

print('Data types:')
print(df_nmd1_long.dtypes)
print(f'\nMissing values in Rate column: {df_nmd1_long["Rate"].isna().sum()}')
print(f'Non-missing values in Rate column: {df_nmd1_long["Rate"].notna().sum()}')
print('\nFinal cleaned NMD1 sample:')
print(df_nmd1_long.head(8).to_string())

Data types:
Sex        object
Year        int64
Rate      float64
Source     object
dtype: object

Missing values in Rate column: 0
Non-missing values in Rate column: 354

Final cleaned NMD1 sample:
       Sex  Year       Rate Source
0    Males  1907  26.694265   NMD1
1  Females  1907   5.222553   NMD1
2  Persons  1907  16.897077   NMD1
3    Males  1908  25.347353   NMD1
4  Females  1908   4.666333   NMD1
5  Persons  1908  15.915600   NMD1
6    Males  1909  25.010777   NMD1
7  Females  1909   5.307562   NMD1


## 3. Cleaning NMD 5: Remoteness area by sex (2001–2024)

NMD 5 has the same wide structure as NMD 1, but with three identifier columns 
instead of two: Remoteness area, Sex and Measure.

One thing the exploration phase revealed: the Sex column has trailing whitespace 
(`'Males   '` instead of `'Males'`). This is invisible in a spreadsheet but 
causes filter operations to silently fail in pandas. We fix it with `.str.strip()`.

In [10]:
df_nmd5_raw = pd.read_excel(FILE_PATH, sheet_name='Table NMD 5', header=1)

print('Raw shape:', df_nmd5_raw.shape)
print('\nRemotenss area values:')
print(df_nmd5_raw['Remoteness area'].unique())
print('\nSex values (note potential trailing spaces):')
print(repr(df_nmd5_raw['Sex'].unique()))  # repr() shows spaces explicitly
print('\nMeasure values:')
print(df_nmd5_raw['Measure'].unique())

Raw shape: (65, 27)

Remotenss area values:
['Major Cities' 'Inner Regional' 'Outer Regional' 'Remote' 'Very Remote'
 'Total Australia'
 'n.p. not available for publication but included in totals where applicable, unless otherwise indicated.'
 nan 'Notes'
 '1. Data cells with small values have been randomly assigned to protect the confidentiality of individuals. As a result, some totals will not equal the sum of their components. Cells with a zero value have not been affected by confidentialisation. '
 '2. Deaths are counted according to year the death was registered by the Registries of Births, Deaths and Marriages (RBDM), not necessarily the year in which the death occurred.'
 '3. Crude rates are per 100,000 population as at 30 June of the same year. '
 '4. Age-standardised rates are per 100,000 standard population as detailed in the Explanatory notes.'
 "5. Rates based on small numbers are volatile and unreliable. Age-standardised death rates based on less than 20 deaths and crude r

In [11]:
# Fix trailing whitespace in Sex column BEFORE any filtering
# .str.strip() removes leading and trailing spaces from string columns
# This is one of the most common data quality issues in government spreadsheets
df_nmd5_raw['Sex'] = df_nmd5_raw['Sex'].str.strip()
df_nmd5_raw['Remoteness area'] = df_nmd5_raw['Remoteness area'].str.strip()

print('Sex values after stripping whitespace:')
print(repr(df_nmd5_raw['Sex'].unique()))

Sex values after stripping whitespace:
array(['Males', 'Females', 'Persons', nan], dtype=object)


In [12]:
# Filter to age-standardised rates only
# Exclude 'Total Australia' remoteness rows — we have this already in NMD1
# and keeping it here would create duplication in any combined analysis
df_nmd5 = df_nmd5_raw[
    (df_nmd5_raw['Measure'] == 'Age-standardised rate (per 100,000)') &
    (df_nmd5_raw['Remoteness area'] != 'Total Australia')
].copy()

# Melt to long format — same pattern as NMD1 but with an extra id column
id_cols_5 = ['Remoteness area', 'Sex', 'Measure']
year_cols_5 = [c for c in df_nmd5.columns if isinstance(c, int)]

df_nmd5_long = pd.melt(
    df_nmd5,
    id_vars=id_cols_5,
    value_vars=year_cols_5,
    var_name='Year',
    value_name='Rate'
)

# Handle suppression codes and fix types
for code in SUPPRESSION_CODES:
    df_nmd5_long['Rate'] = df_nmd5_long['Rate'].replace(code, np.nan)

df_nmd5_long['Rate'] = pd.to_numeric(df_nmd5_long['Rate'], errors='coerce')
df_nmd5_long['Year'] = df_nmd5_long['Year'].astype(int)
df_nmd5_long = df_nmd5_long.drop(columns=['Measure'])
df_nmd5_long['Source'] = 'NMD5'

print('Cleaned NMD5 shape:', df_nmd5_long.shape)
print(f'Missing Rate values: {df_nmd5_long["Rate"].isna().sum()}')
print('\nSample rows:')
print(df_nmd5_long.head(8).to_string())

Cleaned NMD5 shape: (360, 5)
Missing Rate values: 48

Sample rows:
  Remoteness area      Sex  Year  Rate Source
0    Major Cities    Males  2001  18.6   NMD5
1  Inner Regional    Males  2001  22.4   NMD5
2  Outer Regional    Males  2001  25.4   NMD5
3          Remote    Males  2001  22.7   NMD5
4     Very Remote    Males  2001  44.8   NMD5
5    Major Cities  Females  2001   5.2   NMD5
6  Inner Regional  Females  2001   5.6   NMD5
7  Outer Regional  Females  2001   5.9   NMD5


/var/folders/69/qktlcbyn66z87k613hl1zt600000gn/T/ipykernel_9180/3371736788.py:23: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_nmd5_long['Rate'] = df_nmd5_long['Rate'].replace(code, np.nan)


## 4. Cleaning NMD 13a: Children and young people (2001–2024)

NMD 13a covers suicide among people under 25, broken into age bands. 
The `'n.p.'` suppression codes are particularly common here because small 
counts in young age groups frequently fall below the publishable threshold. 
Remember: these are NOT zeros, they represent real but uncountable events.

In [13]:
df_nmd13_raw = pd.read_excel(FILE_PATH, sheet_name='Table NMD 13a', header=1)

print('Raw shape:', df_nmd13_raw.shape)
print('\nAge group values:')
print(df_nmd13_raw['Age group'].unique())
print('\nMeasure values:')
print(df_nmd13_raw['Measure'].unique())

Raw shape: (28, 26)

Age group values:
['0–14' '5–14' '5–17' '15–17' '18–24' '0–24' '5–24'
 'n.p. not available for publication but included in totals where applicable, unless otherwise indicated.'
 nan 'Notes'
 '1. Deaths are counted according to year the death was registered by the Registries of Births, Deaths and Marriages (RBDM), not necessarily the year in which the death occurred.'
 "2. Rates and proportions based on small numbers are volatile and unreliable. Age-specific rates and proportions based on a very low death count are not published and appear as 'n.p.'. "
 '3. In its reports, the Australian Bureau of Statistics (ABS) combines data on suicide at ages 5 to 14 years. For more information about Australian mortality data, including scope and coverage of the collection and a quality declaration, please refer to Causes of death, Australia available from the ABS website.'
 '4. Deaths registered between 2006 and 2021 are based on the final version of cause of death data; deaths

In [14]:
# NMD13a uses 'Age-specific rate' rather than 'Age-standardised rate'
# because these are narrow age bands — age-standardisation isn't applied
# within a single 5-year age group. This is the correct measure to use here.
df_nmd13 = df_nmd13_raw[
    df_nmd13_raw['Measure'] == 'Age-specific rate (per 100,000)'
].copy()

id_cols_13 = ['Age group', 'Measure']
year_cols_13 = [c for c in df_nmd13.columns if isinstance(c, int)]

df_nmd13_long = pd.melt(
    df_nmd13,
    id_vars=id_cols_13,
    value_vars=year_cols_13,
    var_name='Year',
    value_name='Rate'
)

for code in SUPPRESSION_CODES:
    df_nmd13_long['Rate'] = df_nmd13_long['Rate'].replace(code, np.nan)

df_nmd13_long['Rate'] = pd.to_numeric(df_nmd13_long['Rate'], errors='coerce')
df_nmd13_long['Year'] = df_nmd13_long['Year'].astype(int)
df_nmd13_long = df_nmd13_long.drop(columns=['Measure'])
df_nmd13_long['Source'] = 'NMD13'

print('Cleaned NMD13 shape:', df_nmd13_long.shape)
print(f'Suppressed (n.p.) values converted to NaN: {df_nmd13_long["Rate"].isna().sum()}')
print('\nSample rows:')
print(df_nmd13_long.head(8).to_string())

Cleaned NMD13 shape: (168, 4)
Suppressed (n.p.) values converted to NaN: 2

Sample rows:
  Age group  Year  Rate Source
0      0–14  2001   0.3  NMD13
1      5–14  2001   0.4  NMD13
2      5–17  2001   1.7  NMD13
3     15–17  2001   6.1  NMD13
4     18–24  2001  15.9  NMD13
5      0–24  2001   5.3  NMD13
6      5–24  2001   6.6  NMD13
7      0–14  2002   0.2  NMD13


/var/folders/69/qktlcbyn66z87k613hl1zt600000gn/T/ipykernel_9180/1774690867.py:20: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_nmd13_long['Rate'] = df_nmd13_long['Rate'].replace(code, np.nan)


## 5. Cleaning NMD 16: Deaths of despair (1997–2024)

NMD 16 combines three causes of death: suicide, alcoholic liver disease/cirrhosis
and accidental poisoning (drug-related). This "deaths of despair" framework 
comes from public health research examining mortality linked to social and 
economic distress rather than purely biological causes.

The structure here has three identifier columns: Selected causes of death, Sex, Measure.

In [15]:
df_nmd16_raw = pd.read_excel(FILE_PATH, sheet_name='Table NMD 16', header=1)

print('Raw shape:', df_nmd16_raw.shape)
print('\nCause of death values:')
print(df_nmd16_raw['Selected causes of death'].unique())
print('\nSex values:')
print(repr(df_nmd16_raw['Sex'].unique()))

Raw shape: (46, 31)

Cause of death values:
['Alcoholic liver disease and cirrhosis' 'Suicide' 'Accidental poisoning'
 'All combined (Alcoholic liver disease and cirrhosis, Suicide, Accidental poisoning)'
 nan 'Notes'
 '1. Deaths are counted according to year the death was registered by the Registries of Births, Deaths and Marriages (RBDM), not necessarily the year in which the death occurred.'
 '2. Crude rates are per 100,000 population as at 30 June of the same year. '
 '3. Age-standardised rates are per 100,000 standard population as detailed in the Explanatory notes.'
 "4. Rates based on small numbers are volatile and unreliable. Age-standardised death rates based on less than 20 deaths and crude rates based on a very low death count are not published and appear as 'n.p.'. "
 '5. For alcoholic liver disease and cirrhosis, ICD-10 codes are K70, K73–K74.'
 '6. For accidental poisoning, ICD-10 codes are X40–X45, Y10–Y15, Y45, Y47, Y49.'
 '7. For intentional self-harm (suicide), ICD-10

In [16]:
# Strip whitespace from Sex column (same issue as NMD5)
df_nmd16_raw['Sex'] = df_nmd16_raw['Sex'].str.strip()

# Filter to age-standardised rates only
df_nmd16 = df_nmd16_raw[
    df_nmd16_raw['Measure'] == 'Age-standardised rate (per 100,000)'
].copy()

# Create a shorter label for the cause of death to keep charts readable
# The raw strings are very long — we recode them here
cause_labels = {
    'Alcoholic liver disease and cirrhosis': 'Alcohol-related',
    'Suicide': 'Suicide',
    'Accidental poisoning': 'Drug-related',
    'All combined (Alcoholic liver disease and cirrhosis, Suicide, Accidental poisoning)': 'All combined'
}
df_nmd16['Cause'] = df_nmd16['Selected causes of death'].map(cause_labels)

print('Cause labels applied:')
print(df_nmd16[['Selected causes of death', 'Cause']].drop_duplicates().to_string())

Cause labels applied:
                                                              Selected causes of death            Cause
0                                                Alcoholic liver disease and cirrhosis  Alcohol-related
1                                                                              Suicide          Suicide
2                                                                 Accidental poisoning     Drug-related
3  All combined (Alcoholic liver disease and cirrhosis, Suicide, Accidental poisoning)     All combined


In [17]:
id_cols_16 = ['Cause', 'Sex', 'Measure']
year_cols_16 = [c for c in df_nmd16.columns if isinstance(c, (int, float)) 
                or (isinstance(c, str) and c.isdigit())]

# Note: some year columns in NMD16 may be read as strings ('2023', '2024')
# We handle both int and string year columns here
year_cols_16 = [c for c in df_nmd16.columns 
                if c not in id_cols_16 + ['Selected causes of death']
                and str(c).strip().isdigit()]

df_nmd16_long = pd.melt(
    df_nmd16,
    id_vars=id_cols_16,
    value_vars=year_cols_16,
    var_name='Year',
    value_name='Rate'
)

for code in SUPPRESSION_CODES:
    df_nmd16_long['Rate'] = df_nmd16_long['Rate'].replace(code, np.nan)

df_nmd16_long['Rate'] = pd.to_numeric(df_nmd16_long['Rate'], errors='coerce')
df_nmd16_long['Year'] = pd.to_numeric(df_nmd16_long['Year'], errors='coerce').astype(int)
df_nmd16_long = df_nmd16_long.drop(columns=['Measure'])
df_nmd16_long['Source'] = 'NMD16'

print('Cleaned NMD16 shape:', df_nmd16_long.shape)
print('\nSample rows:')
print(df_nmd16_long.head(8).to_string())

Cleaned NMD16 shape: (336, 5)

Sample rows:
             Cause      Sex  Year       Rate Source
0  Alcohol-related    Males  1997   9.052593  NMD16
1          Suicide    Males  1997  23.716000  NMD16
2     Drug-related    Males  1997   2.909464  NMD16
3     All combined    Males  1997  35.678057  NMD16
4  Alcohol-related  Females  1997   3.138877  NMD16
5          Suicide  Females  1997   6.235357  NMD16
6     Drug-related  Females  1997   1.608558  NMD16
7     All combined  Females  1997  10.982792  NMD16


## 6. Cleaning NMD 17: Psychosocial risk factors (2017–2024)

NMD 17 is already in long format, the most analysis-ready table in this dataset. 
However it needs three specific fixes:

1. **Trailing whitespace** in the Sex column (same issue as NMD5 and NMD16)
2. **ICD-10 Z-code separation**, the Risk factor column contains both the code 
   and description joined together. We split these into two columns.
3. **Proportion as percentage**, the Proportion column represents percentage of 
   deaths with this risk factor, e.g. 45.7 means 45.7%. We'll keep it as-is 
   but rename it clearly.

### Why separating the ICD-10 Z-codes matters

The Z-codes are part of ICD-10 Chapter 21, covering 'Factors influencing health 
status and contact with health services.' Unlike disease codes (which describe 
what killed someone), Z-codes describe the circumstances of their life. Z915 
(prior self-harm) is a clinical risk marker. Z635 (family disruption from 
separation/divorce) is a social determinant. Z604 (social isolation) bridges both. 
Having these as separate columns lets us sort, filter and label charts cleanly.

In [18]:
df_nmd17_raw = pd.read_excel(FILE_PATH, sheet_name='Table NMD 17', header=1)

print('Raw shape:', df_nmd17_raw.shape)
print('\nColumns:', df_nmd17_raw.columns.tolist())
print('\nFirst 5 rows:')
print(df_nmd17_raw.head(5).iloc[:, :6].to_string())

Raw shape: (1350, 7)

Columns: ['Risk factor ', 'Frequency rank', 'Year', 'Sex', 'Age group', 'Number', 'Proportion']

First 5 rows:
                                                                                    Risk factor  Frequency rank    Year       Sex Age group  Number
0                                                             Z915 Personal history of self-harm              1  2017.0  Females       0–24    53.0
1                                            Z635 Disruption of family by separation and divorce              2  2017.0  Females       0–24    17.0
2                                           Z630 Problems in relationship with spouse or partner              3  2017.0  Females       0–24    14.0
3                Z634 Disappearance or death of family member or member of primary support group              4  2017.0  Females       0–24    11.0
4  Z614 Problems related to alleged sexual abuse of child by person within primary support group              5  2017.0  Female

In [19]:
df_nmd17 = df_nmd17_raw.copy()

# Fix 1: Strip whitespace from Sex column
df_nmd17['Sex'] = df_nmd17['Sex'].str.strip()

# Fix 2: Rename columns for clarity
df_nmd17 = df_nmd17.rename(columns={
    'Risk factor ':     'Risk_factor_raw',  # note trailing space in original
    'Frequency rank':   'Rank',
    'Age group':        'Age_group',
    'Number':           'Deaths_with_factor',
    'Proportion':       'Pct_with_factor'
})

# Fix 3: Split ICD-10 Z-code from description
# The raw string looks like: 'Z915 Personal history of self-harm'
# str.split(n=1) splits on the FIRST space only, giving us ['Z915', 'Personal history...']
# expand=True creates two new columns from the split result
df_nmd17[['ICD10_code', 'Risk_factor_label']] = (
    df_nmd17['Risk_factor_raw']
    .str.strip()
    .str.split(' ', n=1, expand=True)
)

# Fix 4: Remove rows where Year, Sex, or Age_group is NaN
# (these are blank rows at the bottom of the Excel sheet)
df_nmd17 = df_nmd17.dropna(subset=['Year', 'Sex', 'Age_group'])

# Fix 5: Ensure Year is integer
df_nmd17['Year'] = df_nmd17['Year'].astype(int)

# Keep only the columns we need for analysis
df_nmd17 = df_nmd17[[
    'ICD10_code', 'Risk_factor_label', 'Rank', 'Year',
    'Sex', 'Age_group', 'Deaths_with_factor', 'Pct_with_factor'
]]
df_nmd17['Source'] = 'NMD17'

print('Cleaned NMD17 shape:', df_nmd17.shape)
print('\nSex values after cleaning:')
print(df_nmd17['Sex'].unique())
print('\nSample: ICD-10 code split result:')
print(df_nmd17[['ICD10_code', 'Risk_factor_label']].drop_duplicates().head(8).to_string())
print('\nFull sample of cleaned NMD17:')
print(df_nmd17.head(6).to_string())

Cleaned NMD17 shape: (1344, 9)

Sex values after cleaning:
['Females' 'Males']

Sample: ICD-10 code split result:
  ICD10_code                                                                         Risk_factor_label
0       Z915                                                             Personal history of self-harm
1       Z635                                            Disruption of family by separation and divorce
2       Z630                                           Problems in relationship with spouse or partner
3       Z634                Disappearance or death of family member or member of primary support group
4       Z614  Problems related to alleged sexual abuse of child by person within primary support group
5       Z631                                         Problems in relationship with parents and in-laws
6       Z604                                                Social isolation, exclusion, and rejection
7       Z608                                                  

## 7. Final validation: a sanity check on all five tables

Before moving to analysis, we do a brief sanity check on each cleaned dataframe. 
This is a habit worth building: always confirm your cleaned data makes sense 
before you start drawing conclusions from it. A bug in cleaning can produce 
charts that look plausible but are factually wrong.

In [20]:
# Summary of all cleaned dataframes
cleaned_tables = {
    'NMD1  (national trend)':        df_nmd1_long,
    'NMD5  (remoteness)':            df_nmd5_long,
    'NMD13 (youth)':                 df_nmd13_long,
    'NMD16 (deaths of despair)':     df_nmd16_long,
    'NMD17 (risk factors)':          df_nmd17,
}

print(f'{"Table":<35} {"Rows":>6} {"Cols":>5} {"Missing Rate":>14} {"Year range"}')
print('-' * 80)

for name, df in cleaned_tables.items():
    rows, cols = df.shape
    # Rate/value column name differs between tables
    rate_col = 'Rate' if 'Rate' in df.columns else 'Pct_with_factor'
    missing = df[rate_col].isna().sum() if rate_col in df.columns else 'N/A'
    year_range = f"{df['Year'].min()}–{df['Year'].max()}" if 'Year' in df.columns else 'N/A'
    print(f'{name:<35} {rows:>6} {cols:>5} {str(missing):>14} {year_range}')

Table                                 Rows  Cols   Missing Rate Year range
--------------------------------------------------------------------------------
NMD1  (national trend)                 354     4              0 1907–2024
NMD5  (remoteness)                     360     5             48 2001–2024
NMD13 (youth)                          168     4              2 2001–2024
NMD16 (deaths of despair)              336     5              0 1997–2024
NMD17 (risk factors)                  1344     9              0 2017–2024


In [21]:
# Spot check: does NMD1 give sensible national rates for recent years?
# We expect Male rates around 18-20 per 100,000 and Female rates around 6 per 100,000
# based on known Australian epidemiology
print('NMD1 — most recent 3 years, age-standardised rates by sex:')
recent = df_nmd1_long[df_nmd1_long['Year'] >= 2022].sort_values(['Sex', 'Year'])
print(recent[['Sex', 'Year', 'Rate']].to_string(index=False))

print('\nNMD5 — most recent year (2024), rates by remoteness area (Males):')
remote_check = df_nmd5_long[
    (df_nmd5_long['Year'] == 2024) & 
    (df_nmd5_long['Sex'] == 'Males')
].sort_values('Rate', ascending=False)
print(remote_check[['Remoteness area', 'Sex', 'Year', 'Rate']].to_string(index=False))

NMD1 — most recent 3 years, age-standardised rates by sex:
    Sex  Year  Rate
Females  2022   6.2
Females  2023   6.0
Females  2024   5.5
  Males  2022  19.2
  Males  2023  18.4
  Males  2024  18.3
Persons  2022  12.6
Persons  2023  12.1
Persons  2024  11.8

NMD5 — most recent year (2024), rates by remoteness area (Males):
Remoteness area   Sex  Year  Rate
    Very Remote Males  2024  39.6
         Remote Males  2024  39.2
 Outer Regional Males  2024  28.7
 Inner Regional Males  2024  24.9
   Major Cities Males  2024  15.2


## 8. Summary

All five tables are now clean and in long format. The key transformations applied were:

**For NMD 1, 5, 13, 16:** `pd.melt()` to convert from wide (years as columns) 
to long format (one row per observation), filter to age-standardised or 
age-specific rates only, suppression codes replaced with `NaN`.

**For NMD 5, 16:** Additional `.str.strip()` to remove trailing whitespace 
from the Sex column, invisible in a spreadsheet, but causes silent filter 
failures in pandas.

**For NMD 17:** ICD-10 Z-codes split from plain-English labels using 
`str.split(n=1, expand=True)`, columns renamed for clarity, blank rows dropped.

**Next step:** `03_analysis_visualisation.ipynb`: generate the five core charts 
and write the analytical narrative for each one.